In [12]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, get_dataset_config_names
from collections import Counter

dataset_name = "GEM/wiki_auto_asset_turk"
raw_train_data = load_dataset(dataset_name,"wiki_auto_asset_turk",split="train[:5000]")
special_tokens = ["<PAD>","<SOS>","<EOS>","<UNK>"]
modif_matrix_multiplication = "true"



### Helper

In [13]:

# --------------------------------------------------
# Prepare Small Dataset
# --------------------------------------------------

def prepare_small_dataset(
    raw_data,
    max_length=20,
    dataset_size=2000
):

    sentence_pairs = []

    for example in raw_data:

        source = example["source"].strip()
        target = example["target"].strip()

        if source == "" or target == "":
            continue

        source_tokens = source.split()
        target_tokens = target.split()

        if (
            len(source_tokens) <= max_length
            and len(target_tokens) <= max_length
        ):

            sentence_pairs.append(
                (source, target)
            )

    small_dataset = sentence_pairs[:dataset_size]

    return small_dataset

# --------------------------------------------------
# Dataset Context Preparation
# --------------------------------------------------

def prepare_dataset_context(sentence_pairs,d_model,max_vocab_size=3000,seed=42):

    token_counter = Counter()
    for source, target in sentence_pairs:
        source_tokens = source.split()
        target_tokens = target.split()
        token_counter.update(source_tokens)
        token_counter.update(target_tokens)
        
    most_common_tokens = [
        token
        for token, count in token_counter.most_common(
            max_vocab_size - len(special_tokens)
        )
    ]

    vocabulary = special_tokens + most_common_tokens

    token_to_id = {
        token: token_id
        for token_id, token in enumerate(vocabulary)
    }
    id_to_token = {
        token_id: token
        for token, token_id in token_to_id.items()
    }

    vocab_size = len(vocabulary)

    np.random.seed(seed)
    E = np.random.randn(
        d_model,
        vocab_size
    ) * 0.01

    context = {
        "special_tokens": special_tokens,
        "vocabulary": vocabulary,
        "token_to_id": token_to_id,
        "id_to_token": id_to_token,
        "vocab_size": vocab_size,
        "d_model": d_model,
        "E": E,
        "pad_id": token_to_id["<PAD>"],
        "sos_id": token_to_id["<SOS>"],
        "eos_id": token_to_id["<EOS>"],
        "unk_id": token_to_id["<UNK>"]
    }
    return context

# --------------------------------------------------
# Softmax
# --------------------------------------------------
def softmax(x):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# --------------------------------------------------
# Layer Normalization
# --------------------------------------------------
def layer_norm(x, eps=1e-5):
    mean = np.mean(x, axis=0, keepdims=True)
    variance = np.var(x, axis=0, keepdims=True)
    normalized = (x - mean) / np.sqrt(variance + eps)
    return normalized

# --------------------------------------------------
# Tokenization and Embedding
# --------------------------------------------------
def prepare_input(sentence,context,add_sos=False,add_eos=False):

    token_to_id = context["token_to_id"]
    vocab_size = context["vocab_size"]
    E = context["E"]

    tokens = sentence.split()
    if add_sos:
        tokens = ["<SOS>"] + tokens
    if add_eos:
        tokens = tokens + ["<EOS>"]
    token_ids = []
    for token in tokens:
        token_id = token_to_id.get(token,token_to_id["<UNK>"])
        token_ids.append(token_id)
    one_hot_vectors = []
    for token_id in token_ids:
        one_hot = np.zeros( (vocab_size, 1))
        one_hot[token_id, 0] = 1
        one_hot_vectors.append(one_hot)

    embedded_tokens = []
    for one_hot in one_hot_vectors:
        embedded_token = matrix_multiply(E,one_hot)
        embedded_tokens.append(embedded_token)
    X = np.hstack(embedded_tokens)
    return tokens, token_ids, X



def matrix_multiply(A, B):

    if modif_matrix_multiplication != "true":
        return A @ B

    rows_A = A.shape[0]
    cols_A = A.shape[1]

    rows_B = B.shape[0]
    cols_B = B.shape[1]

    result = np.zeros((rows_A, cols_B))

    for i in range(rows_A):
        for j in range(cols_B):
            for k in range(cols_A):
                tmp = A[i, k] * B[k, j]
                result[i, j] = tmp + result[i, j]

    return result

# --------------------------------------------------
# Initialize Encoder Block Parameters
# --------------------------------------------------
# --------------------------------------------------
# Attention Parameters
# --------------------------------------------------
def initialize_attention_parameters(d_model,num_heads,seed=42):
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads")
    np.random.seed(seed)
    d_head = d_model // num_heads
    W_Q_heads = []
    W_K_heads = []
    W_V_heads = []
    for head_index in range(num_heads):
        W_Q_heads.append(np.random.randn(d_head,d_model) * 0.01)
        W_K_heads.append(np.random.randn(d_head,d_model) * 0.01)
        W_V_heads.append(np.random.randn(d_head,d_model) * 0.01)
    W_O = np.random.randn(d_model,d_model) * 0.01
    return {
        "num_heads": num_heads,
        "d_head": d_head,
        "W_Q_heads": W_Q_heads,
        "W_K_heads": W_K_heads,
        "W_V_heads": W_V_heads,
        "W_O": W_O
    } 

# --------------------------------------------------
# FFN Parameters
# --------------------------------------------------
def initialize_ffn_parameters(d_model,d_ff,seed=42):
    np.random.seed(seed)
    W_1 = np.random.randn(d_ff,d_model) * 0.01
    b_1 = np.zeros((d_ff,1))
    W_2 = np.random.randn(d_model,d_ff) * 0.01
    b_2 = np.zeros((d_model,1))
    return {
        "W_1": W_1,
        "b_1": b_1,
        "W_2": W_2,
        "b_2": b_2
    }

def initialize_encoder_block_parameters(d_model,d_ff,num_heads,seed=42):
    return {
        "self_attention": initialize_attention_parameters(d_model,num_heads,seed),
        "ffn": initialize_ffn_parameters(d_model,d_ff,seed + 100)
    }

def initialize_decoder_block_parameters(d_model,d_ff,num_heads,seed=42):
    return {
        "self_attention": initialize_attention_parameters(d_model,num_heads,seed),
        "cross_attention": initialize_attention_parameters(d_model,num_heads,seed + 50),
        "ffn": initialize_ffn_parameters(d_model,d_ff,seed + 100)
    }

def initialize_output_parameters(d_model,vocab_size,seed=42):
    np.random.seed(seed)
    W_vocab = np.random.randn(vocab_size,d_model) * 0.01
    b_vocab = np.zeros((vocab_size,1))
    return {
        "W_vocab": W_vocab,
        "b_vocab": b_vocab
    }

def cross_entropy_loss(probabilities,target_ids):
    eps = 1e-12
    total_loss = 0
    for position,target_id in enumerate(target_ids):
        token_probability = probabilities[target_id,position]
        total_loss += -np.log(token_probability + eps)
    loss = total_loss / len(target_ids)
    return loss

# --------------------------------------------------
# Token Accuracy
# --------------------------------------------------
def token_accuracy(predicted_ids,target_ids):
    correct = 0
    for predicted_id,target_id in zip(predicted_ids,target_ids):
        if predicted_id == target_id:
            correct += 1
    accuracy = correct / len(target_ids)
    return accuracy

# --------------------------------------------------
# IDs to Tokens
# --------------------------------------------------
def ids_to_tokens(token_ids,context):
    id_to_token = context["id_to_token"]
    tokens = []
    for token_id in token_ids:
        tokens.append(id_to_token[int(token_id)])
    return tokens


def vocabulary_projection(decoder_output,output_parameters):
    W_vocab = output_parameters["W_vocab"]
    b_vocab = output_parameters["b_vocab"]
    logits = matrix_multiply(W_vocab,decoder_output) + b_vocab
    exp_logits = np.exp(logits - np.max(logits,axis=0,keepdims=True))
    probabilities = exp_logits / np.sum(exp_logits,axis=0,keepdims=True)

    predicted_ids = np.argmax(probabilities,axis=0)
    return logits,probabilities,predicted_ids

### Multi-Head Self-Attention

In [14]:
# --------------------------------------------------
# Multi-Head Attention
# --------------------------------------------------
def multi_head_attention(query_input,key_value_input,attention_parameters,use_mask=False):
    num_heads = attention_parameters["num_heads"]
    d_head = attention_parameters["d_head"]
    W_Q_heads = attention_parameters["W_Q_heads"]
    W_K_heads = attention_parameters["W_K_heads"]
    W_V_heads = attention_parameters["W_V_heads"]
    W_O = attention_parameters["W_O"]
    head_outputs = []
    attention_weights_all_heads = []
    for head_index in range(num_heads):
        W_Q = W_Q_heads[head_index]
        W_K = W_K_heads[head_index]
        W_V = W_V_heads[head_index]
        Q = matrix_multiply(W_Q,query_input)
        K = matrix_multiply(W_K,key_value_input)
        V = matrix_multiply(W_V,key_value_input)
        scores = matrix_multiply(Q.T,K) / np.sqrt(d_head)
        if use_mask:
            sequence_length = scores.shape[0]
            mask = np.triu(np.ones((sequence_length,sequence_length)),k=1)
            scores = np.where(mask == 1,-np.inf,scores)
        attention_weights = softmax(scores)
        head_output = matrix_multiply(V,attention_weights.T)
        head_outputs.append(head_output)
        attention_weights_all_heads.append(attention_weights)
    multi_head_output = np.vstack(head_outputs)
    output = matrix_multiply(W_O,multi_head_output)
    return output,attention_weights_all_heads

### FFN

In [15]:
# --------------------------------------------------
# Feed-Forward Network
# --------------------------------------------------
def feed_forward_network(X,parameters):
    W_1 = parameters["W_1"]
    b_1 = parameters["b_1"]
    W_2 = parameters["W_2"]
    b_2 = parameters["b_2"]
    hidden = matrix_multiply(W_1,X) + b_1
    hidden = np.maximum(0,hidden)
    output = matrix_multiply(W_2,hidden) + b_2
    return output

### Encoder And Decoder Block

In [16]:
# --------------------------------------------------
# Encoder Block
# --------------------------------------------------
def encoder_block(X,parameters,norm_type="post"):
    self_attention_parameters = parameters["self_attention"]
    ffn_parameters = parameters["ffn"]
    if norm_type == "post":
        attention_output,attention_weights = multi_head_attention(X,X,self_attention_parameters,use_mask=False)
        norm1_output = layer_norm(X + attention_output)
        ffn_output = feed_forward_network(norm1_output,ffn_parameters)
        encoder_output = layer_norm(norm1_output + ffn_output)
    elif norm_type == "pre":
        norm_input = layer_norm(X)
        attention_output,attention_weights = multi_head_attention(norm_input,norm_input,self_attention_parameters,use_mask=False)
        residual1_output = X + attention_output
        norm2_input = layer_norm(residual1_output)
        ffn_output = feed_forward_network(norm2_input,ffn_parameters)
        encoder_output = residual1_output + ffn_output
    else:
        raise ValueError("norm_type must be 'post' or 'pre'")
    return encoder_output,attention_weights



# --------------------------------------------------
# Decoder Block
# --------------------------------------------------
def decoder_block(decoder_X,encoder_output,parameters,norm_type="post"):
    self_attention_parameters = parameters["self_attention"]
    cross_attention_parameters = parameters["cross_attention"]
    ffn_parameters = parameters["ffn"]
    if norm_type == "post":
        self_attention_output,self_attention_weights = multi_head_attention(decoder_X,decoder_X,self_attention_parameters,use_mask=True)
        norm1_output = layer_norm(decoder_X + self_attention_output)
        cross_attention_output,cross_attention_weights = multi_head_attention(norm1_output,encoder_output,cross_attention_parameters,use_mask=False)
        norm2_output = layer_norm(norm1_output + cross_attention_output)
        ffn_output = feed_forward_network(norm2_output,ffn_parameters)
        decoder_output = layer_norm(norm2_output + ffn_output)
    elif norm_type == "pre":
        norm_input = layer_norm(decoder_X)
        self_attention_output,self_attention_weights = multi_head_attention(norm_input,norm_input,self_attention_parameters,use_mask=True)
        residual1_output = decoder_X + self_attention_output
        norm2_input = layer_norm(residual1_output)
        cross_attention_output,cross_attention_weights = multi_head_attention(norm2_input,encoder_output,cross_attention_parameters,use_mask=False)
        residual2_output = residual1_output + cross_attention_output
        norm3_input = layer_norm(residual2_output)
        ffn_output = feed_forward_network(norm3_input,ffn_parameters)
        decoder_output = residual2_output + ffn_output
    else:
        raise ValueError("norm_type must be 'post' or 'pre'")
    return decoder_output,self_attention_weights,cross_attention_weights

### Encoder and Decoder Stacking

In [17]:
# --------------------------------------------------
# Initialize Encoder Stack and Encoder Stack
# --------------------------------------------------

def encoder_stack( X,num_layers,d_model,d_ff,num_heads,seed=42, norm_type="post"):

    encoder_parameters = []
    for layer_index in range(num_layers):
        layer_parameters = initialize_encoder_block_parameters(d_model=d_model,d_ff=d_ff,num_heads=num_heads,seed=seed + layer_index)
        encoder_parameters.append(layer_parameters)

    encoder_output = X
    attention_weights_all_layers = []
    for layer_parameters in encoder_parameters:
        encoder_output, attention_weights = encoder_block(encoder_output, layer_parameters,norm_type=norm_type)
        attention_weights_all_layers.append(attention_weights)

    return encoder_output, attention_weights_all_layers


# --------------------------------------------------
# Initialize Decoder Stack and Decoder Stacking
# --------------------------------------------------
def decoder_stack(decoder_X, encoder_output, num_layers,d_model,d_ff,num_heads,seed=42,norm_type="post"):
    decoder_parameters = []
    for layer_index in range(num_layers):
        decoder_parameters.append(
            initialize_decoder_block_parameters(d_model,d_ff,num_heads,seed + layer_index)
        )
    decoder_output = decoder_X
    decoder_self_attention_weights = []
    decoder_cross_attention_weights = []
    for layer_parameters in decoder_parameters:
        decoder_output,self_weights,cross_weights = decoder_block(decoder_output,encoder_output,layer_parameters,norm_type)
        decoder_self_attention_weights.append(self_weights)
        decoder_cross_attention_weights.append(cross_weights)
    return decoder_output,decoder_self_attention_weights,decoder_cross_attention_weights

### Model

In [18]:
d_model = 8
small_dataset = prepare_small_dataset(
    raw_train_data,
    max_length=20,
    dataset_size=2000
)

context = prepare_dataset_context(
    small_dataset,
    d_model=d_model,
    max_vocab_size=3000
)
# --------------------------------------------------
# One Dataset Example
# --------------------------------------------------
source_sentence,target_sentence = small_dataset[0]
encoder_tokens,encoder_ids,encoder_X = prepare_input(source_sentence,context)
decoder_tokens,decoder_ids,decoder_X = prepare_input(target_sentence,context,add_sos=True)
target_tokens,target_ids,target_X = prepare_input(target_sentence,context,add_eos=True)

# --------------------------------------------------
# Architecture
# --------------------------------------------------
d_model = context["d_model"]
d_ff = 16
num_heads = 2
num_encoder_layers = 2
num_decoder_layers = 2
norm_type = "post"

# --------------------------------------------------
# Initialize Encoder and Decoder
# --------------------------------------------------
encoder_output, encoder_attention_weights = encoder_stack(encoder_X, num_layers=num_encoder_layers,d_model=d_model,
                                                          d_ff=d_ff,num_heads=num_heads,seed=42,
                                                          norm_type="post")
decoder_output,decoder_self_attention_weights,decoder_cross_attention_weights = decoder_stack(decoder_X, encoder_output, 
                                                                                num_layers=num_encoder_layers,d_model=d_model,d_ff=d_ff,
                                                                                num_heads=num_heads,seed=42,norm_type="post")

output_parameters = initialize_output_parameters(d_model,context["vocab_size"],seed=200)



# --------------------------------------------------
# Output Projection
# --------------------------------------------------

logits,probabilities,predicted_ids = vocabulary_projection(decoder_output,output_parameters)

In [19]:
# --------------------------------------------------
# Check Shapes
# --------------------------------------------------
print("encoder_X:",encoder_X.shape)
print("encoder_output:",encoder_output.shape)
print("decoder_X:",decoder_X.shape)
print("decoder_output:",decoder_output.shape)
print("target length:",len(target_ids))


# --------------------------------------------------
# Loss and Accuracy
# --------------------------------------------------
loss = cross_entropy_loss(probabilities,target_ids)
accuracy = token_accuracy(predicted_ids,target_ids)
predicted_tokens = ids_to_tokens(predicted_ids,context)


# --------------------------------------------------
# Result
# --------------------------------------------------
print("Source sentence:")
print(source_sentence)

print("Target sentence:")
print(target_sentence)

print("Decoder input tokens:")
print(decoder_tokens)

print("Expected output tokens:")
print(target_tokens)

print("Predicted output tokens:")
print(predicted_tokens)

print("Loss:",loss)
print("Token accuracy:",accuracy)

encoder_X: (8, 18)
encoder_output: (8, 18)
decoder_X: (8, 21)
decoder_output: (8, 21)
target length: 21
Source sentence:
The Confederation of African Football or CAF is the administrative and controlling body for African association football .
Target sentence:
The Confederation of African Football often referred to as just CAF , is the organization that controls African football .
Decoder input tokens:
['<SOS>', 'The', 'Confederation', 'of', 'African', 'Football', 'often', 'referred', 'to', 'as', 'just', 'CAF', ',', 'is', 'the', 'organization', 'that', 'controls', 'African', 'football', '.']
Expected output tokens:
['The', 'Confederation', 'of', 'African', 'Football', 'often', 'referred', 'to', 'as', 'just', 'CAF', ',', 'is', 'the', 'organization', 'that', 'controls', 'African', 'football', '.', '<EOS>']
Predicted output tokens:
['station', 'Lord', 'Catani', 'championship', 'motorway', 'KG-99', 'show', 'Symphoniker', 'London', 'Maryland', 'being', 'da', 'flats', 'services', 'novels', '